In [ ]:
print("Hello World");

Hello World


In [2]:
# Cell 1: GPU check + imports
import torch
import os
import shutil
import yaml
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=== YOLOv8 Training Setup ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available - training will be very slow!")

=== YOLOv8 Training Setup ===
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory: 4.3 GB


In [ ]:
# Cell 2: Install ultralytics and train YOLOv8n-seg
!pip install ultralytics

from ultralytics import YOLO

# Project paths
project_root = Path().absolute().parent
data_dir = project_root / "data"
models_dir = project_root / "models"

# Ensure directories exist
models_dir.mkdir(exist_ok=True)

# Load YOLOv8n-seg model
model = YOLO('yolov8n-seg.pt')

# Training configuration
data_yaml_path = data_dir / "indianfoodnet_yolo" / "data.yaml"

print(f"Training data path: {data_yaml_path}")
if not data_yaml_path.exists():
    print(f"ERROR: {data_yaml_path} not found!")
    print("Please ensure the IndianFoodNet dataset is properly set up.")
else:
    # Train the model
    results = model.train(
        data=str(data_yaml_path),
        epochs=60,
        imgsz=640,
        batch=8,
        patience=15,
        device=0 if torch.cuda.is_available() else 'cpu',
        workers=2,
        amp=True,
        save_period=10,
        project=str(models_dir / "runs"),
        name="yolo_indian_seg"
    )

In [ ]:
# Cell 3: Copy best.pt to models/yolov8_indian_seg.pt
import glob

# Find the best.pt file from the latest run
runs_dir = models_dir / "runs" / "yolo_indian_seg"
if runs_dir.exists():
    # Get all run directories and sort them
    run_dirs = [d for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith('train')]
    if run_dirs:
        latest_run = max(run_dirs, key=lambda x: x.stat().st_mtime)
        best_pt_path = latest_run / "weights" / "best.pt"
        
        if best_pt_path.exists():
            # Copy to models directory
            target_path = models_dir / "yolov8_indian_seg.pt"
            shutil.copy2(best_pt_path, target_path)
            print(f"✅ Copied best model to: {target_path}")
            print(f"Source: {best_pt_path}")
        else:
            print(f"ERROR: best.pt not found in {latest_run / 'weights'}")
    else:
        print("ERROR: No training runs found")
else:
    print(f"ERROR: Training directory {runs_dir} not found")

In [ ]:
# Cell 4: Save class names to models/class_names.json
data_yaml_path = data_dir / "indianfoodnet_yolo" / "data.yaml"

if data_yaml_path.exists():
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    class_names = data_config.get('names', {})
    print(f"Found {len(class_names)} classes:")
    for idx, name in class_names.items():
        print(f"  {idx}: {name}")
    
    # Save to JSON
    class_names_path = models_dir / "class_names.json"
    with open(class_names_path, 'w') as f:
        json.dump(class_names, f, indent=2)
    
    print(f"\n✅ Saved class names to: {class_names_path}")
else:
    print(f"ERROR: {data_yaml_path} not found!")

In [ ]:
# Cell 5: Plot training loss curves
import pandas as pd
import matplotlib.pyplot as plt

# Find results.csv from the latest run
runs_dir = models_dir / "runs" / "yolo_indian_seg"
if runs_dir.exists():
    run_dirs = [d for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith('train')]
    if run_dirs:
        latest_run = max(run_dirs, key=lambda x: x.stat().st_mtime)
        results_csv = latest_run / "results.csv"
        
        if results_csv.exists():
            # Read results
            df = pd.read_csv(results_csv)
            
            # Plot training metrics
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            
            # Loss plots
            axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
            axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss')
            axes[0, 0].set_title('Box Loss')
            axes[0, 0].legend()
            axes[0, 0].grid(True)
            
            axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Cls Loss')
            axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Cls Loss')
            axes[0, 1].set_title('Classification Loss')
            axes[0, 1].legend()
            axes[0, 1].grid(True)
            
            axes[1, 0].plot(df['epoch'], df['train/dfl_loss'], label='Train DFL Loss')
            axes[1, 0].plot(df['epoch'], df['val/dfl_loss'], label='Val DFL Loss')
            axes[1, 0].set_title('DFL Loss')
            axes[1, 0].legend()
            axes[1, 0].grid(True)
            
            # Metrics
            axes[1, 1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
            axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
            axes[1, 1].set_title('Detection Metrics')
            axes[1, 1].legend()
            axes[1, 1].grid(True)
            
            plt.tight_layout()
            plt.show()
            
            # Print final metrics
            final_metrics = df.iloc[-1]
            print("\n=== Final Training Metrics ===")
            print(f"Final mAP50: {final_metrics['metrics/mAP50(B)']:.4f}")
            print(f"Final mAP50-95: {final_metrics['metrics/mAP50-95(B)']:.4f}")
            print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
            print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
        else:
            print(f"ERROR: results.csv not found in {latest_run}")
    else:
        print("ERROR: No training runs found")
else:
    print(f"ERROR: Training directory {runs_dir} not found")